In [0]:
%sql

-- VISTA LÓGICA DE NORMALIZACIÓN Y HOMOLOGACIÓN 
-- Se utilizan los atributo tags y titulo para inferir el tipo de producto 
-- Se agrega el atributo tienda

CREATE OR REPLACE VIEW wine_catalog.bronze.productos_clean AS

SELECT 
    *,
    COALESCE(tipo, tipo_inferido) AS categoria
FROM (
select 
CASE 
  WHEN id RLIKE '^[0-9]+$' THEN CAST(id AS BIGINT)
  ELSE NULL
END AS id,
titulo,
CASE 
  WHEN bodega = 'Château Mouton-Rothschild' THEN 'Château Mouton Rothschild'
  WHEN bodega = 'Tenuta dell Ornellaia' THEN 'Tenuta dell''Ornellaia'
  WHEN bodega IN ('Paco y Lola', 'Paco & Lola') THEN 'Paco & Lola'
  WHEN bodega ='Bodega Norton' THEN 'Norton'
  when bodega= 'Tienda Susana Balbo' Then 'Susana Balbo'
  ELSE bodega
END AS bodega,
CASE 
  WHEN tipo LIKE 'Vino%' OR tipo LIKE 'vino%'OR tipo LIKE 'Wine'
    THEN 'Vino'
  WHEN tipo LIKE 'Espumante%' 
    THEN 'Espumante'
  WHEN tipo LIKE 'Grappa' OR tipo LIKE 'Pisco' OR tipo LIKE 'Vodka' OR tipo LIKE 'Spirits' 
    THEN 'Destilado'
  WHEN tipo LIKE 'Aceite%' OR tipo LIKE 'Mieles' 
    THEN 'Delicatessen'
  WHEN tipo LIKE 'Cajas Mix%' OR tipo LIKE 'Bebidas alcohólicas%'
    THEN 'Otros'
END AS tipo,
CASE 
  WHEN tags LIKE '%Vinos%' OR tags LIKE '%Tintos%'
    OR tags LIKE '%vinos%' OR tags LIKE '%tintos%'
    OR tags LIKE '%Blancos%' OR tags LIKE '%Rosado%' 
    OR tags LIKE '%Malbec%' OR tags LIKE '%Cabernet%' 
    OR tags LIKE '%Crios%' OR tags LIKE '%denuestracava%'
    THEN 'Vino'
  WHEN tags LIKE '%Espumosos%' 
    THEN 'Espumante'
  WHEN tags LIKE '%Vermouth%' OR tags LIKE '%Vermut%' OR tags LIKE '%Vermú%' 
    THEN 'Vermut'
  WHEN tags LIKE '%Destilados%' OR tags LIKE '%Whisky%' 
    OR tags LIKE '%Ron%' OR tags LIKE '%Mezcal%' 
    OR tags LIKE '%grappa%' OR tags LIKE '%Digestivo%'
    THEN 'Destilado'
  WHEN tags LIKE '%Catas%' OR tags LIKE '%Cursos%' THEN 'Sin clasificar'
  WHEN tags LIKE '%elicatessen%'  or  tags LIKE '%eilicatessen%' THEN 'Delicatessen'
  WHEN titulo LIKE '%ermouth%' OR titulo LIKE '%ermut%' 
  THEN 'Vermut'
  ELSE 'Sin clasificar'
END AS tipo_inferido,
tags,
CASE
  WHEN precio RLIKE '^[0-9]+(\.[0-9]+)?$' THEN CAST(precio AS DOUBLE)
  ELSE NULL
END as precio,
CASE
  WHEN precio_tachado RLIKE '^[0-9]+(\.[0-9]+)?$' THEN CAST(precio_tachado AS DOUBLE)
  ELSE NULL
END as precio_tachado,
TRY_CAST(stock AS BOOLEAN) AS stock,
publicado,
descripcion,
CASE 
    WHEN moneda = 'ARS' AND bodega LIKE '%Balbo%' THEN 'Susana Balbo'
    WHEN moneda = 'ARS' AND bodega LIKE '%Norton%' THEN 'Norton'
    WHEN moneda = 'UYU' THEN 'Ocio Wine'
    WHEN moneda = 'EUR' THEN 'Exclusivas Miro'
    WHEN moneda = 'USD' THEN 'La Barrica'
END AS tienda,
pais,
moneda
from wine_catalog.bronze.products
where pais IN ('Argentina','USA','Uruguay','España') and precio IS NOT NULL
)

--De 2.201 registros originales, 1.894 resultan válidos tras filtrar IDs corruptos.


In [0]:
%sql

-- DISTRIBUCIÓN DE VALORES NULOS EN CATEGORIA
SELECT

    count(*) as Registros_Validos,
    count(*) - count(c.CATEGORIA) as Nulos
    from wine_catalog.bronze.productos_clean c
order by 2 desc

-- Se recupero el 100% de registros con tipo Nulo mediante inferencia desde tags y titulo (1021 registros).

Registros_Validos,Nulos
1894,0


In [0]:
%sql
-- DATASET GLOBAL DE PRODUCTOS VITIVINÍCOLAS 

CREATE OR REPLACE VIEW wine_catalog.bronze.productos_vino AS
SELECT *
FROM wine_catalog.bronze.productos_clean
WHERE categoria IN ('Vino', 'Espumante', 'Vermut')


-- Contiene únicamente categorías relevantes para el análisis de mercado (Vino, Espumante y Vermut). 
-- Se excluyen 42 productos que corresponden a merchandising, experiencias y no vitivinícolas.


In [0]:
%sql
-- ESTADISTICAS DE PRECIOS POR CATEGORÍA Y MONEDA

SELECT 
    categoria,
    moneda,
    COUNT(*) as registros,
    ROUND(MIN(precio), 2) as minimo,
    ROUND(MAX(precio), 2) as maximo,
    ROUND(AVG(precio), 2) as promedio,
    ROUND(PERCENTILE(precio, 0.01), 2) as p01,
    ROUND(PERCENTILE(precio, 0.25), 2) as p25,
    ROUND(PERCENTILE(precio, 0.50), 2) as mediana,
    ROUND(PERCENTILE(precio, 0.75), 2) as p75,
    ROUND(PERCENTILE(precio, 0.99), 2) as p99
FROM wine_catalog.bronze.productos_vino
WHERE moneda IN ('ARS', 'UYU')
GROUP BY moneda, categoria
order by moneda, categoria;

-- El análisis de precios y bodegas se limita a Argentina y Uruguay, evaluando cada mercado en su moneda local y 
-- sin conversión cambiaria. 

categoria,moneda,registros,minimo,maximo,promedio,p01,p25,mediana,p75,p99
Espumante,ARS,10,8900.0,25700.0,13948.0,8900.0,8935.0,13020.0,17000.0,25007.0
Vino,ARS,182,5400.0,1588025.0,127923.88,5497.2,16170.0,52119.0,143341.0,1159190.32
Vermut,UYU,3,700.0,700.0,700.0,700.0,700.0,700.0,700.0,700.0
Vino,UYU,83,435.6,22000.0,4109.17,435.6,1432.0,2300.0,3987.5,22000.0


In [0]:
%sql

-- VERIFICACION DE POSIBLES OUTLIERS 

SELECT titulo, bodega, precio, moneda, categoria
FROM wine_catalog.bronze.productos_vino
where moneda IN ('ARS', 'UYU')
AND precio > (
    SELECT PERCENTILE(precio, 0.99)
    FROM wine_catalog.bronze.productos_vino
    WHERE moneda IN ('ARS', 'UYU')
)
ORDER BY precio DESC

-- Los outliers no son errores de carga, los precios extremos corresponden a productos ultra premium (ediciones 
-- especiales, colecciones de añadas múltiples). Se optó por segmentar en lugar de descartar.


titulo,bodega,precio,moneda,categoria
2014 Doble Magnum Nosotros Single Vineyard Nómade Malbec,Susana Balbo,1588025.0,ARS,Vino
2017 Doble Magnum Nosotros Single Vineyard Nómade Malbec,Susana Balbo,1260625.0,ARS,Vino
Nosotros Terroir Library Collection 99 pts 2013 - 2016 - 2017 - 2019,Susana Balbo,1135397.0,ARS,Vino


In [0]:
%sql

-- DATASET ANALÍTICO REGIONAL DE PRODUCTOS VITIVINÍCOLAS 
-- Se crea segmentación de precios por cuartiles
 
CREATE OR REPLACE VIEW wine_catalog.bronze.precios_regional AS
WITH percentiles AS (
    SELECT 
        moneda,
        categoria,
        PERCENTILE(precio, 0.25) AS p25,
        PERCENTILE(precio, 0.75) AS p75
    FROM wine_catalog.bronze.productos_vino
    WHERE moneda IN ('ARS', 'UYU')
    GROUP BY moneda, categoria
)
SELECT 
    p.titulo,
    p.bodega,
    p.categoria,
    p.moneda,
    p.precio,
    p.pais,
    p.tags,
    CASE
        WHEN p.precio <= per.p25 THEN 'Económico'
        WHEN p.precio <= per.p75 THEN 'Medio'
        ELSE 'Premium'
    END AS segmento_precio,
    p.stock
FROM wine_catalog.bronze.productos_vino p
LEFT JOIN percentiles per
    ON p.moneda = per.moneda
    AND p.categoria = per.categoria
WHERE p.moneda IN ('ARS', 'UYU')


    


In [0]:
%sql
select * from wine_catalog.bronze.precios_regional 

titulo,bodega,categoria,moneda,precio,pais,tags,segmento_precio,stock
2017 Doble Magnum Nosotros Single Vineyard Nómade Malbec,Susana Balbo,Vino,ARS,1260625.0,Argentina,"Alta Gama, Cosecha vigente, denuestracava, Envío Gratis, Guardados, Malbec, Nosotros, Puntaje 98",Premium,false
2014 Doble Magnum Nosotros Single Vineyard Nómade Malbec,Susana Balbo,Vino,ARS,1588025.0,Argentina,"Alta Gama, Cosecha vigente, denuestracava, Envío Gratis, Guardados, Malbec, Nosotros, Puntaje 98",Premium,false
Nosotros Terroir Library Collection 99 pts 2013 - 2016 - 2017 - 2019,Susana Balbo,Vino,ARS,1135397.0,Argentina,"denuestracava, Guardados, Malbec, Nosotros, Puntaje 99",Premium,true
2012 Nosotros 25 años Library Edition Cabernet Sauvignon,Susana Balbo,Vino,ARS,411349.0,Argentina,"Alta Gama, CabernetSauvignon, denuestracava, Envío Gratis, Guardados, Nosotros",Premium,true
Recomendados Mix Osadía de Crear Alta Gama,Susana Balbo,Vino,ARS,73342.0,Argentina,"SameDay, SN",Medio,true
2020 Susana Balbo Signature Barrel Fermented Chardonnay,Susana Balbo,Vino,ARS,44802.0,Argentina,"Blancos y Rosados, Chardonnay, Cosecha vigente, denuestracava, Envío Gratis Cajas, especial gourmet, Guardados, SN",Medio,true
Recomendados Mix Rosados de Susana,Susana Balbo,Vino,ARS,90202.0,Argentina,"Crios, Rosé",Medio,true
Benmarco Plata Cabernet Sauvignon,Susana Balbo,Vino,ARS,80431.0,Argentina,"CabernetSauvignon, SameDay",Medio,true
Vertical BenMarco Expresivo Gualtallary 2019-2020-2021,Susana Balbo,Vino,ARS,200529.0,Argentina,"Alta Gama, denuestracava, Envío Gratis, Guardados, SN",Premium,false
Frapera Crios,Susana Balbo,Vino,ARS,83853.0,Argentina,"Crios, SN",Medio,true
